In [2]:
%pip install faiss-cpu

   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   --- ------------------------------------ 1.6/16.5 MB 7.7 MB/s eta 0:00:02
   -------- ------------------------------- 3.4/16.5 MB 9.2 MB/s eta 0:00:02
   ------------- -------------------------- 5.5/16.5 MB 8.9 MB/s eta 0:00:02
   --------------- ------------------------ 6.6/16.5 MB 7.7 MB/s eta 0:00:02
   ------------------- -------------------- 8.1/16.5 MB 8.0 MB/s eta 0:00:02
   ----------------------- ---------------- 9.7/16.5 MB 7.7 MB/s eta 0:00:01
   ---------------------------- ----------- 11.8/16.5 MB 8.0 MB/s eta 0:00:01
   ---------------------------------- ----- 14.4/16.5 MB 8.5 MB/s eta 0:00:01
   ---------------------------------------- 16.5/16.5 MB 8.7 MB/s  0:00:02
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from google import genai
import os
from dotenv import load_dotenv

import faiss
import numpy as np

In [2]:
!python --version

Python 3.11.9


In [3]:
#  SETUP GEMINI

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)


In [ ]:
# LOAD MULTIPLE TXT DOCUMENTS

data_dir = "documents"
document = ""

for file_name in os.listdir(data_dir):

    if file_name.endswith(".txt"):
        
        file_path = os.path.join(data_dir, file_name)

        with open(file_path, "r", encoding="utf-8") as f:
            document += f.read() + "\n\n"

        print("Document loaded:", file_name)

print("Total characters:", len(document))

Document loaded: EMP-001_Employment_Agreement_Alpha_Technologies.txt
Document loaded: EMP-002_Employment_Agreement_Beta_Systems.txt
Document loaded: NDA-001_Mutual_NDA.txt
Document loaded: NDA-002_One_Way_NDA.txt
Document loaded: SVC-001_Software_Services_Agreement.txt
Document loaded: SVC-002_Software_Services_Agreement_Revised.txt
Document loaded: VND-001_Vendor_Supply_Agreement.txt
Document loaded: VND-002_Vendor_Supply_Agreement_Revised.txt
Total characters: 15316


In [4]:
print(type(document))
print("Document characters:", len(document))

<class 'str'>
Document characters: 15316


In [ ]:
chunk_size = 500
chunk_overlap = 50

chunks = []
start = 0

while start < len(document):
    end = start + chunk_size
    chunk = document[start:end]
    chunks.append(chunk)
    start = end - chunk_overlap

print("Total chunks:", len(chunks))

Total chunks: 35


In [6]:
result = client.models.embed_content(
    model = "gemini-embedding-001",
    contents=chunks
)


chunk_embeddings = np.array(
    [embedding.values for embedding in result.embeddings],
    dtype="float32"
)

print("Embedding shape:", chunk_embeddings.shape)

Embedding shape: (35, 3072)


In [7]:
dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print("Vectors stored in Faiss:", index.ntotal)

Vectors stored in Faiss: 35


In [8]:
def embed_question(question):

    result = client.models.embed_content(
        model = "gemini-embedding-001",
        contents=[question]
    )

    question_embedding = np.array(
        [result.embeddings[0].values],
        dtype = "float32"
    )

    return question_embedding

In [9]:
def search_faiss(question_embedding, k=3):
    distances, indices = index.search(
        question_embedding,
        k=k
    )

    return indices[0]

In [10]:
def retrieve_chunks(indices):
    retrieved_chunks =[]

    for i in indices:
        retrieved_chunks.append(chunks[i])

    return retrieved_chunks

In [11]:
def create_context(retrieved_chunks):
    context ='\n\n'.join(retrieved_chunks)

    return context

In [ ]:
#  GENERATE ANSWER

def generate_answer(question, context):

    prompt = f"""
You are a legal document RAG assistant.

Answer the question ONLY using the provided context.

Rules:
1. Do not use outside knowledge.
2. Do not make assumptions.
3. If the answer cannot be found in the context, say:
"I don't know based on the provided document."
4. Give a concise and accurate answer.
5. If the context contains a document ID or section name, include it as the source.
6. Do not provide legal advice.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""


    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text




In [13]:
# COMPLETE RAG PIPELINE

'''def answer_question(question, k=3):

    question_embedding = embed_question(question)

    indices = search_faiss(
        question_embedding,
        k=k
    )

    retrieved_chunks = retrieve_chunks(indices)

    print("\n" + "=" * 60)
    print("RETRIEVED CHUNKS")
    print("=" * 60)

    for i, chunk in enumerate(retrieved_chunks):

        print(f"\n--- Chunk {i+1} ---")
        print(chunk)

    context = create_context(retrieved_chunks)

    answer = generate_answer(
        question,
        context
    )

    return answer'''


'def answer_question(question, k=3):\n\n    question_embedding = embed_question(question)\n\n    indices = search_faiss(\n        question_embedding,\n        k=k\n    )\n\n    retrieved_chunks = retrieve_chunks(indices)\n\n    print("\n" + "=" * 60)\n    print("RETRIEVED CHUNKS")\n    print("=" * 60)\n\n    for i, chunk in enumerate(retrieved_chunks):\n\n        print(f"\n--- Chunk {i+1} ---")\n        print(chunk)\n\n    context = create_context(retrieved_chunks)\n\n    answer = generate_answer(\n        question,\n        context\n    )\n\n    return answer'

In [16]:
# QUESTION 1

'''question = "What is the notice period in EMP-001?"

answer = answer_question(question)

print(answer)
'''

'question = "What is the notice period in EMP-001?"\n\nanswer = answer_question(question)\n\nprint(answer)\n'

In [17]:
# QUESTION 1

'''question = "Which contract has the longest termination notice period?"

answer = answer_question(question)

print(answer)'''


'question = "Which contract has the longest termination notice period?"\n\nanswer = answer_question(question)\n\nprint(answer)'

In [ ]:
question = "What is the termination notice period?"

context = """
Document ID: EMP-001
Section 6 - TERMINATION

Either party may terminate employment without cause by providing
30 days' written notice.
"""

answer = generate_answer(question, context)

print(answer) 

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


The termination notice period is 30 days' written notice. 

Source: Document ID: EMP-001, Section 6 - TERMINATION


In [22]:
question = "What are the services, fees and payment of software services agreement?"

context = """
Document ID: SVC-001
Software Services Agreement

Section 1 - SERVICES
Provider will provide application development, maintenance, monitoring,
and technical support described in applicable statements of work.

Section 2 - FEES AND PAYMENT
Customer will pay USD 18,000 per month.
Invoices are due within 30 days of receipt.
Disputed amounts must be identified within 15 days.
"""
answer = generate_answer(question, context)

print(answer)

Based on Document ID: SVC-001 (Software Services Agreement):

* **Services (Section 1):** Application development, maintenance, monitoring, and technical support as described in applicable statements of work.
* **Fees and Payment (Section 2):** 
  * **Fees:** USD 18,000 per month.
  * **Payment Terms:** Invoices are due within 30 days of receipt. Any disputed amounts must be identified within 15 days.
